In [1]:
import time
import asyncpg
from app.data.dto.main.SeaPort import SeaPortDB

db_name="BunkeringBot"
db_user="def_user"
db_password="super_password"
db_host="77.37.96.222"
db_port="5432"

connection_pool = await asyncpg.create_pool(
            database=db_name,
            user=db_user,
            password=db_password,
            host=db_host,
            port=db_port,
            min_size=1,
            max_size=20
        )


async with connection_pool.acquire() as conn:
    rows = await conn.fetch(
        """
        SELECT *
        FROM public.ports_vector_new
        """
    )

    for r in rows:
        p = SeaPortDB.from_db_row(r)
        ports.append(p)

len(ports)

In [9]:
ports = []

async with connection_pool.acquire() as conn:
    rows = await conn.fetch(
        """
        SELECT *
        FROM public.ports_vector_new
        """
    )

    for r in rows:
        p = SeaPortDB.from_db_row(r)
        ports.append(p)

len(ports)

9267

In [15]:
import pandas as pd

df = pd.DataFrame([p.model_dump() for p in ports])

In [16]:
df.head()

,locode,country_code,country_name,port_name,latitude,longitude,rank_score,similarity_score,combined_score,match_type,mabux_ids,port_size,mabux_id,barge_status,truck_status,agent_contact_list,bubble_id,search_key,id
0,BHGBQ,,Bahrain,Al Muharraq,26.249620,50.610165,None,None,None,unknown,"[137, 138]",tiny,NaN,None,None,None,1713117933382x489210049983866750,al muharraq|bhgbq|bahrain|,43e9cb02-1386-4f2c-90a5-181760785475
1,BHMAN,,Bahrain,Manamah,26.218846,50.593182,None,None,None,unknown,[137],tiny,NaN,None,None,None,1713117933382x839598483197600400,manamah|bhman|bahrain|,bfd5844b-c1e5-4083-8bd8-7663938bd8cd
2,BHMIN,,Bahrain,Mina Sulman Port,26.189335,50.608705,None,None,None,unknown,[137],tiny,NaN,None,None,None,1713117933383x491979747041934700,mina sulman port|bhmin|bahrain|,4e144495-69c1-4f93-8878-3a0541aba333
3,BHBAH,,Bahrain,Bahrain,26.266667,50.633333,None,None,None,unknown,"[137, 138]",tiny,137.0,None,None,None,1713117933382x945141392676871300,bahrain|bhbah|bahrain|137,76a1c5c9-9bc1-4ab6-b571-1d1a4f14b7e2
4,JPOIR,,Japan,Okushiri,42.176800,139.523250,None,None,None,unknown,[455],tiny,NaN,None,None,None,1713117928626x274270081242699260,okushiri|jpoir|japan|,f61b83d4-b647-4461-b445-a774d56779dd


In [41]:
import pandas as pd

# Select and prepare columns
columns = [
    'locode', 'country_name', 'port_name', 'port_size',
    'latitude', 'longitude', 'barge_status',
    'truck_status', 'agent_contact_list'
]

data = df[columns].copy()

# Insert manual column
data.insert(4, 'manual_size_status', None)

# Rename to nice column names
data = data.rename(columns={
    'locode': 'UN/LOCODE',
    'country_name': 'Country',
    'port_name': 'Port Name',
    'port_size': 'Port Size (Auto)',
    'manual_size_status': 'Port Size (Manual)',
    'latitude': 'Latitude',
    'longitude': 'Longitude',
    'barge_status': 'Barge Available',
    'truck_status': 'Truck Available',
    'agent_contact_list': 'Agent Contacts'
})

# Sort and reset index
data = data.sort_values(by='Country').reset_index(drop=True)

# Make rows full width when displayed
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

data.head()

# Export to Excel
data.to_excel("Sea ports with manual size input.xlsx", index=False)


In [8]:
ports = []

async with connection_pool.acquire() as conn:
    rows = await conn.fetch(
        """
        SELECT *
        FROM public.ports_vector_new
        WHERE latitude > 90 or latitude < -90 or longitude > 180 or longitude < -180;
        """
    )

    for r in rows:
        #p = SeaPortDB.from_db_row(r)
        ports.append(r)

len(ports)

0

In [ ]:
ports[0]['locode']

In [3]:
from app.services.external_api.searoute_api import SearouteApi
searoute_api = SearouteApi("https://api.searoutes.com/", "EWhTo2x2hihNDCPZjCaMgFDWGegJoVLnYP7mqi5L")

found = []
not_found = []

for port in ports:
    s_port, err = await searoute_api.get_port_info(port['locode'].strip())
    if s_port and not err:
        found.append(s_port)
    else:
        not_found.append(port)

    time.sleep(0.7)

In [4]:
len(found)

8

In [5]:
from dotenv import load_dotenv
load_dotenv()
from app.services.db_service import DbService
sql_db_service = DbService()
await sql_db_service.init_pool()

r, err = await sql_db_service.bulk_upsert_ports(found)

In [7]:
len(r)

8

In [ ]:
ports = []


async with connection_pool.acquire() as conn:
    rows = await conn.fetch(
        """
        SELECT *
        FROM public.ports_vector_new
        WHERE 1 = 1
            AND port_size = 'large'
            AND (
                mabux_id IS NOT NULL
                OR (mabux_ids IS NOT NULL AND cardinality(mabux_ids) > 0)
            )
        """
    )

    for r in rows:
        p = SeaPortDB.from_db_row(r)
        ports.append(p)


In [ ]:
len(ports)

In [ ]:
from app.services.external_api.searoute_api import SearouteApi
searoute_api = SearouteApi("https://api.searoutes.com/", "EWhTo2x2hihNDCPZjCaMgFDWGegJoVLnYP7mqi5L")

In [ ]:
import geopandas as gpd

gdf = gpd.read_file("./data/IFR_LOCATION_PORTS.geojson")

In [ ]:
import pandas as pd

df = pd.read_csv("./data/UpdatedPub150.csv")
df['Harbor Size'].unique()
large_df = df[df['Harbor Size'] == 'Large'][['UN/LOCODE', "Latitude", "Longitude", 'Harbor Size']][:10]


for i, locode in zip(large_df.index, large_df['UN/LOCODE'].values):
    searoute_port, err = await searoute_api.get_port_info(locode.strip().replace(" ", ""))
    if searoute_port and not err:
        large_df.loc[i, "Searoute port size"] = searoute_port.size

In [ ]:
large_df

In [ ]:
import json
with open("./data/somewhere_from.json") as fp:
    t = json.load(fp)

In [ ]:
len(t)

In [ ]:
t[0]

In [ ]:
from dotenv import load_dotenv
load_dotenv()
from app.services.db_service import DbService
db = DbService()
await db.init_pool()


In [ ]:
import pandas as pd

SIZE_MAP = {
    "L": "large",
    "M": "medium",
    "S": "small",
    "V": "tiny",
}

rows = []

for port in t[:15]:
    lat = port["LATITUDE"]
    lon = port["LONGITUDE"]
    t_size = port.get("HARBORSIZE")

    ports_db, err = await db.search_ports_nearby(lat, lon)
    if err or not ports_db:
        continue

    db_port = ports_db[0]
    locode = db_port.locode

    searoute_port, err = await searoute_api.get_port_info(locode.strip().replace(" ", ""))
    searoute_size = searoute_port.size if searoute_port and not err else None

    rows.append({
        "locode": locode,
        "db_port_size": SIZE_MAP.get(db_port.port_size, db_port.port_size),
        "searoute_port_size": SIZE_MAP.get(searoute_size, searoute_size),
        "t_object_port_size": SIZE_MAP.get(t_size, t_size),
    })

df = pd.DataFrame(rows)
df


In [ ]:
supabase_df = pd.read_csv("./data/Supabase Snippet All Ports.csv")
supabase_df['locode'] = supabase_df['locode'].apply(lambda x : x.upper())
large_supabase_ports = supabase_df[supabase_df['size'] == 'large']
large_supabase_ports

import time

updating = []
for locode in large_supabase_ports['locode'].values:
    searoute_port, err = await searoute_api.get_port_info(locode.strip().replace(" ", ""))
    if searoute_port and not err:
        updating.append(searoute_port)
    time.sleep(1)



In [ ]:
r, er = await db.bulk_upsert_ports(updating)


In [ ]:
er

In [ ]:
len(r)